## Analisis Afiliados de la SP

In [19]:
import requests
import io
import pandas as pd

url = 'https://www.spensiones.cl/inf_estadistica/series_afp/afiliados/afiliados.xls'
r = requests.get(url, timeout=30)

df = pd.read_excel(io.BytesIO(r.content), sheet_name=0,header=1, engine='xlrd')

df = df.rename(columns={df.columns[0]: 'fecha'})

df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')

df.dropna(subset=['fecha'])

df.replace('-',pd.NA)

display(df.tail(5))


,fecha,SISTEMA,ALAMEDA,APORTA,APORTA FOMENTA,ARMONIZA,BANGUARDIA,BANNUESTRA,BANSANDER,CAPITAL,...,PROVIDA,QUALITAS,QUALITAS.1,SAN CRISTOBAL,SANTA MARIA,SUMMA,SUMMA BANSANDER,UNION,VALORA,UNO
536,2026-01-01,12131027,-,-,-,-,-,-,-,1417994,...,2541552,-,-,-,-,-,-,-,-,1517738
537,2026-02-01,12149568,-,-,-,-,-,-,-,1415328,...,2529379,-,-,-,-,-,-,-,-,1557994
538,NaT,Informe Estadistico de Afiliados y Cotizantes ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
539,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
540,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
from playwright.sync_api import sync_playwright
import pandas as pd
from io import StringIO

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()

    # Step 1: land on the actual stats section (where user navigates from)
    page.goto(
        "https://www.spensiones.cl/apps/centroEstadisticas/paginaCuadrosCCEE.php"
        "?menu=sci&menuN1=afil&menuN2=afp",
        wait_until="networkidle"
    )

    print("Cookies after landing page:")
    for cookie in page.context.cookies():
        print(cookie)

    # Step 2: now follow the real chain
    page.goto(
        "https://www.spensiones.cl/apps/loadEstadisticas/siSP.php"
        "?id=inf_estadistica/aficot/trimestral/2025/12/02F.html"
        "&menu=sci&menuN1=afil&menuN2=afp&orden=30&ext=.html",
        wait_until="networkidle"
    )

    print("\nFinal URL:", page.url)
    print("Status: check above — if 404 page will say so")

    html = page.content()
    browser.close()

# Step 3: parse
tables = pd.read_html(StringIO(html))
print(f"\n{len(tables)} tables found")
for i, df in enumerate(tables):
    print(f"\n--- Table {i} ---")
    print(df.head(3))

ModuleNotFoundError: No module named 'playwright'